# 04. Data Validation for BI Star Schema

**Goal:** Validate the integrity of the `goldBI.duckdb` Star Schema and perform specific "Dashboard Mirror Tests" to ensure the data aligns perfectly with the expected Power BI visualizations.

In [50]:
import duckdb
import pandas as pd

# Connect to gold layer
con = duckdb.connect('../data/gold/goldBI.duckdb')
print("Connected to goldBI ")

Connected to goldBI 


## 1. Architectural Integrity Validations

In [51]:
# ── Row count validation ──────────────────────────────────────────────
bi_count = con.execute("SELECT COUNT(*) FROM bi_data").fetchone()[0]
fact_count = con.execute("SELECT COUNT(*) FROM fact_loans").fetchone()[0]

print(f"BI Parquet rows:  {bi_count:,}")
print(f"Fact table rows:  {fact_count:,}")
print(f"Difference:       {bi_count - fact_count:,}")
print(f"Match: {bi_count == fact_count}")
assert bi_count == fact_count, "Row mismatch between BI parquet and fact table!"

BI Parquet rows:  2,257,158
Fact table rows:  2,257,158
Difference:       0
Match: True


In [52]:
# ── Duplicate validation ──────────────────────────────────────────────
dupes = con.execute("""
    SELECT loan_id, COUNT(*) as count
    FROM fact_loans
    GROUP BY loan_id
    HAVING COUNT(*) > 1
""").fetchdf()

print(f"Duplicate loan_ids in fact table: {len(dupes)}")
assert len(dupes) == 0, "Duplicate primary keys found!"

# ── Null validation for Foreign Keys ──────────────────────────────
nulls = con.execute("""
    SELECT 
        COUNT(*) - COUNT(loan_id) AS loan_id_nulls,
        COUNT(*) - COUNT(risk_key) AS risk_key_nulls,
        COUNT(*) - COUNT(purpose_key) AS purpose_key_nulls,
        COUNT(*) - COUNT(date_key) AS date_key_nulls,
        COUNT(*) - COUNT(addr_state) AS addr_state_nulls,
        COUNT(*) - COUNT(annual_inc_category) AS annual_inc_cat_nulls
    FROM fact_loans
""").fetchdf()

print("\nNull counts in key/calc columns:")
print(nulls)

Duplicate loan_ids in fact table: 0

Null counts in key/calc columns:
   loan_id_nulls  risk_key_nulls  purpose_key_nulls  date_key_nulls  \
0              0               0                  0               0   

   addr_state_nulls  annual_inc_cat_nulls  
0                 0                     0  


## 2. Dimension Consistency Checks

In [53]:
# ── Dimension validation ──────────────────────────────────────────────
print("dim_date:")
print(con.execute("SELECT COUNT(*) as rows, MIN(year) as min_year, MAX(year) as max_year FROM dim_date").fetchdf())

print("\ndim_risk:")
print(con.execute("""
    SELECT 
        COUNT(*) as rows, 
        COUNT(DISTINCT sub_grade) as grades, 
        COUNT(DISTINCT mths_since_recent_inq_label) as inq_labels 
    FROM dim_risk
""").fetchdf())

print("\ndim_purpose:")
print(con.execute("SELECT COUNT(*) as rows FROM dim_purpose").fetchdf())

print("\ndim_geography:")
print(con.execute("SELECT COUNT(*) as rows FROM dim_geography").fetchdf())

dim_date:
   rows  min_year  max_year
0   139      2007      2018

dim_risk:
   rows  grades  inq_labels
0   904      35           4

dim_purpose:
   rows
0    14

dim_geography:
   rows
0    51


## 3. Business Logic Validations (Dashboard Mirror Tests)

In [54]:
# Dashboard Mirror Test 1: Global KPIs
print("--- DASHBOARD MIRROR: GLOBAL KPIs ---")
df = con.execute("""
    SELECT 
        COUNT(*) as total_loans, 
        SUM(CASE WHEN default_flag IS NOT NULL THEN 1 ELSE 0 END) as closed_loans,
        ROUND(AVG(default_flag)*100, 2) as default_rate_pct,
        ROUND(AVG(int_rate), 2) as avg_interest_rate
    FROM fact_loans
""").fetchdf()
print(df)
actual = df['default_rate_pct'].iloc[0]
# Business invariant: LC default rate historically falls between 10-25%
assert 10.0 < actual < 25.0, f"Default rate {actual}% outside expected range [10%, 25%]"
print(f"✓ Default rate {actual}% is within business invariant [10%, 25%]")


--- DASHBOARD MIRROR: GLOBAL KPIs ---
   total_loans  closed_loans  default_rate_pct  avg_interest_rate
0      2257158     1368244.0             21.23               0.13
✓ Default rate 21.23% is within business invariant [10%, 25%]


In [55]:
# Dashboard Mirror Test 2: Geographic Analysis
print("--- DASHBOARD MIRROR: GEOGRAPHIC ANALYSIS ---")
df = con.execute("""
    SELECT 
        g.state_name_usa,
        COUNT(*) as total_loans,
        SUM(CASE WHEN f.default_flag IS NOT NULL THEN 1 ELSE 0 END) as closed_loans,
        ROUND(AVG(f.default_flag)*100, 2) as default_rate_pct
    FROM fact_loans f
    JOIN dim_geography g ON f.addr_state = g.addr_state
    GROUP BY g.state_name_usa
    HAVING COUNT(*) > 100
    ORDER BY default_rate_pct DESC
""").fetchdf()
print(df.head(5))
# Business invariant: No state should have default rate > 40% or < 5%
max_rate = df['default_rate_pct'].max()
min_rate = df['default_rate_pct'].min()
assert max_rate < 40.0, f"Anomalous state default rate: {max_rate}%"
assert min_rate > 5.0, f"Suspiciously low state default rate: {min_rate}%"
print(f"✓ State default rates [{min_rate}%, {max_rate}%] within business invariant")


--- DASHBOARD MIRROR: GEOGRAPHIC ANALYSIS ---
     state_name_usa  total_loans  closed_loans  default_rate_pct
0  Mississippi, USA        12617        6795.0             28.39
1     Nebraska, USA         7803        3678.0             27.03
2     Arkansas, USA        17037       10224.0             25.43
3      Alabama, USA        27233       16922.0             24.97
4     Oklahoma, USA        20648       12497.0             24.79
✓ State default rates [14.27%, 28.39%] within business invariant


In [56]:
# Dashboard Mirror Test 3: Income Monotonicity
print("--- DASHBOARD MIRROR: CUSTOMER ANALYSIS (Income) ---")
df = con.execute("""
    SELECT 
        annual_inc_category,
        COUNT(*) as total_loans,
        SUM(CASE WHEN default_flag IS NOT NULL THEN 1 ELSE 0 END) as closed_loans,
        ROUND(AVG(default_flag)*100, 2) as default_rate_pct
    FROM fact_loans
    GROUP BY annual_inc_category
    ORDER BY default_rate_pct DESC
""").fetchdf()
print(df)
# Business invariant: Low income must have HIGHER default rate than Very High income
low_rate = df.loc[df['annual_inc_category'] == 'Low (<40k)', 'default_rate_pct'].iloc[0]
high_rate = df.loc[df['annual_inc_category'] == 'Very High (>120k)', 'default_rate_pct'].iloc[0]
assert low_rate > high_rate, f"Risk monotonicity broken: Low ({low_rate}%) should > Very High ({high_rate}%)"
print(f"✓ Risk monotonicity confirmed: Low ({low_rate}%) > Very High ({high_rate}%)")


--- DASHBOARD MIRROR: CUSTOMER ANALYSIS (Income) ---
  annual_inc_category  total_loans  closed_loans  default_rate_pct
0          Low (<40k)       349351      214112.0             25.02
1    Medium (40k-80k)      1088382      674692.0             22.17
2     High (80k-120k)       509113      306744.0             18.89
3   Very High (>120k)       310312      172696.0             17.03
✓ Risk monotonicity confirmed: Low (25.02%) > Very High (17.03%)


In [57]:
# --- AUDITOR FIXES: Strict Assertions ---
print("\n--- Checking 'Current' loans presence ---")
current_loans = con.execute("SELECT COUNT(*) FROM fact_loans WHERE default_flag IS NULL").fetchone()[0]
print(f"Préstamos 'Current' en fact_loans: {current_loans:,}")
assert current_loans > 500_000, "Error: Filtro de madurez eliminó la cartera activa."

print("\n--- Checking Geographic Orphan Records ---")
orphan_states = con.execute("""
    SELECT DISTINCT f.addr_state
    FROM fact_loans f
    LEFT JOIN dim_geography g ON f.addr_state = g.addr_state
    WHERE g.addr_state IS NULL
""").fetchdf()
if not orphan_states.empty:
    print(f"Orphan states found: {orphan_states['addr_state'].tolist()}")
assert orphan_states.empty, "Error: Join huérfano en dim_geography."



--- Checking 'Current' loans presence ---
Préstamos 'Current' en fact_loans: 888,914

--- Checking Geographic Orphan Records ---


In [58]:
# ── Referential Integrity Checks (BCBS 239 Compliance) ─────────────
print("\n--- Referential Integrity Checks ---")

# fact_loans -> dim_risk
orphan_risk = con.execute("""
    SELECT COUNT(*) FROM fact_loans f
    LEFT JOIN dim_risk dr ON f.risk_key = dr.risk_key
    WHERE dr.risk_key IS NULL
""").fetchone()[0]
print(f"Orphan risk_keys in fact_loans: {orphan_risk}")
assert orphan_risk == 0, "RI violation: fact_loans -> dim_risk"

# fact_loans -> dim_purpose
orphan_purpose = con.execute("""
    SELECT COUNT(*) FROM fact_loans f
    LEFT JOIN dim_purpose dp ON f.purpose_key = dp.purpose_key
    WHERE dp.purpose_key IS NULL
""").fetchone()[0]
print(f"Orphan purpose_keys in fact_loans: {orphan_purpose}")
assert orphan_purpose == 0, "RI violation: fact_loans -> dim_purpose"

# fact_loans -> dim_date
orphan_date = con.execute("""
    SELECT COUNT(*) FROM fact_loans f
    LEFT JOIN dim_date dd ON f.date_key = dd.date_key
    WHERE dd.date_key IS NULL
""").fetchone()[0]
print(f"Orphan date_keys in fact_loans: {orphan_date}")
assert orphan_date == 0, "RI violation: fact_loans -> dim_date"

print("All referential integrity checks passed ✓")


--- Referential Integrity Checks ---
Orphan risk_keys in fact_loans: 0
Orphan purpose_keys in fact_loans: 0
Orphan date_keys in fact_loans: 0
All referential integrity checks passed ✓


In [59]:
con.close()
print("\nAll validations passed ")


All validations passed 
